# House price models
A fresh pipeline: explore the data, prepare numeric/categorical features, and train a `GradientBoostingRegressor` with a baseline forest for comparison.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
sns.set(style='ticks')


In [ ]:
dataset = pd.read_csv('housing_az_sqm_azn.csv')
dataset = dataset.sample(frac=1.0, random_state=42).reset_index(drop=True)
dataset.head()


In [ ]:
target = 'PriceAZN'
features = dataset.columns.drop(target)
X = dataset[features]
y = dataset[target]
print(f'Observations: {len(dataset)}, features: {len(features)}')


In [ ]:
summary = dataset.describe(include='all').T[['count', 'mean', 'min', 'max']]
categorical_snapshot = dataset.select_dtypes(exclude=[np.number]).nunique().sort_values(ascending=False)
print(summary.head())
print('
Categorical cardinality:')
print(categorical_snapshot)


In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)


In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', one_hot)
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])


In [ ]:
rf_reg = RandomForestRegressor(n_estimators=400, random_state=13, n_jobs=-1)
gb_reg = GradientBoostingRegressor(random_state=13, n_estimators=320, learning_rate=0.07, max_depth=3)

models = {
    'random_forest': rf_reg,
    'gradient_boosting': gb_reg,
}



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

def evaluate(estimator):
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', estimator)
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    return {
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': root_mean_squared_error(y_test, preds),
        'R2': r2_score(y_test, preds),
        'pipeline': pipeline,
    }

results = {name: evaluate(model) for name, model in models.items()}
{key: {k: round(v, 3) for k, v in res.items() if k != 'pipeline'} for key, res in results.items()}


In [ ]:
final_pipe = results['gradient_boosting']['pipeline']
cv_scores = cross_val_score(final_pipe, X, y, cv=5, scoring='r2')
print('CV R2 mean:', cv_scores.mean().round(3))
print('CV R2 std:', cv_scores.std().round(3))


In [ ]:
new_samples = pd.DataFrame({
    'Bedrooms': [2, 4],
    'Bathrooms': [3, 2],
    'Sqm': [56, 120],
    'City': ['Sumqayit', 'Baku'],
})
final_predictions = final_pipe.predict(new_samples)
pd.concat([new_samples, pd.Series(final_predictions, name='PredictedPriceAZN')], axis=1)


In [ ]:
rf_pipe = results['random_forest']['pipeline']
model = rf_pipe.named_steps['model']
onehot = rf_pipe.named_steps['prep'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = onehot.get_feature_names_out(categorical_features)
feature_names = numeric_features + cat_feature_names.tolist()
importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.head(10)


In [ ]:
metrics_rows = []
for name, res in results.items():
    mae = res['MAE'] if isinstance(res['MAE'], (int, float)) else res['MAE']
    rmse = res['RMSE'] if isinstance(res['RMSE'], (int, float)) else res['RMSE']
    r2 = res['R2'] if isinstance(res['R2'], (int, float)) else res['R2']
    metrics_rows.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
metrics_df = pd.DataFrame(metrics_rows).set_index('model')
metrics_df


In [ ]:
gb_pipe = results['gradient_boosting']['pipeline']
test_preds = gb_pipe.predict(X_test)
plt.figure(figsize=(6, 5))
plt.scatter(y_test, test_preds, alpha=0.7, s=25)
lims = [min(y_test.min(), test_preds.min()), max(y_test.max(), test_preds.max())]
plt.plot(lims, lims, color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual PriceAZN')
plt.ylabel('Predicted PriceAZN')
plt.title('Gradient Boosting: actual vs predicted')
plt.tight_layout()


In [ ]:
rf_pipe = results['random_forest']['pipeline']
model = rf_pipe.named_steps['model']
onehot = rf_pipe.named_steps['prep'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = onehot.get_feature_names_out(categorical_features)
feature_names = numeric_features + cat_feature_names.tolist()
importances = pd.Series(model.feature_importances_, index=feature_names)
important = importances.sort_values(ascending=False).head(12)
plt.figure(figsize=(8, 5))
important[::-1].plot(kind='barh', color='steelblue')
plt.xlabel('Importance')
plt.title('Random Forest feature importance (top 12)')
plt.tight_layout()
